# Claims: generated, then tested independently


# Claims: generated with a measured certainty, then tested independently

The goal (2026-10-04): generate new knowledge from existing data, and test it by means separate from the inference that generated it -- with every number measured on genes whose answer was hidden. Design: `instructions/open/63_claims_generate_and_verify.md`; code: `starplast/claims.py`; build: `scripts/build_claims.py`.

Everything below reads the shipped track record (`starplast/data/track_record.parquet`), where every labelled gene was hidden once in folds that never split an orthogroup.

In [1]:
import numpy as np, pandas as pd
from starplast import claims as C, strategies as S, track_record as T
tg = T.shipped('Tg'); tg = tg[tg['target'].astype(str) == 'compartment']
len(tg)

136560

## 1. Leakage between generator and verifier, measured

"Information leakage needs to be quantified not guessed": a signal peptide is location information though it shares no dataset with LOPIT. The shared-mistake ratio: when the generator is wrong, how often the verifier makes the SAME wrong call, over how often it would if the two were independent given the truth (from the verifier's own confusion matrix). 1.0 = independent. A verifier counts as an independent test when the upper end of the 95% bootstrap interval is below `C.INDEPENDENT_BELOW`.

In [2]:
pairs = [(g, v) for g in ('feature_knn', 'supervised_classifier', 'random_forest', 'stacking')
         for v in ('physical_partners', 'structural_homology', 'layer_propagation', 'random_forest', 'feature_knn') if g != v]
m = pd.DataFrame([C.shared_mistakes(tg, g, v) for g, v in pairs])
m[['generator', 'verifier', 'generator_wrong', 'observed', 'expected', 'ratio', 'ratio_low', 'ratio_high']].round(2)

,generator,verifier,generator_wrong,observed,expected,ratio,ratio_low,ratio_high
0,feature_knn,physical_partners,466,0.13,0.11,1.21,0.97,1.47
1,feature_knn,structural_homology,427,0.18,0.14,1.23,1.01,1.47
2,feature_knn,layer_propagation,1210,0.13,0.07,1.91,1.64,2.16
3,feature_knn,random_forest,1490,0.55,0.16,3.53,3.35,3.72
4,supervised_classifier,physical_partners,564,0.11,0.09,1.31,1.07,1.6
5,supervised_classifier,structural_homology,573,0.13,0.09,1.45,1.17,1.72
6,supervised_classifier,layer_propagation,1612,0.16,0.07,2.16,1.94,2.4
7,supervised_classifier,random_forest,2032,0.33,0.09,3.8,3.55,4.06
8,supervised_classifier,feature_knn,1424,0.32,0.1,3.3,3.07,3.54
9,random_forest,physical_partners,518,0.12,0.11,1.1,0.87,1.33


Stacking absorbs every kind of evidence, so nothing can test it independently -- which is why recipes are chosen as the generator that independent evidence CAN test (section 4).

## 2. Leakage between evidence and label, measured

Each evidence family (dataset) alone, held out by orthogroup fold. A family that alone recovers `C.STANDS_OUT` of what all evidence together recovers is flagged for a person to read: a restated label and one dominant biological signal look alike in the numbers. All Toxoplasma compartment labels come from hyperLOPIT (`compartment_source`), so none was predicted from sequence.

In [3]:
rows = []
for org, t in (('Tg', 'compartment'), ('Tg', 'lopit_unified'), ('Pf', 'lopit_pf_location')):
    fr = C.family_recovery(S.Context.shipped(org), t)
    top = fr.iloc[0]
    rows.append({'organism': org, 'label': t, 'all families': fr['all_families'].iloc[0],
                 'strongest family': top['family'], 'its per-class recall': top['per_class'],
                 'share of all': top['share_of_all'], 'flagged': ', '.join(fr[fr.stands_out].family) or 'none'})
pd.DataFrame(rows).round(2)

,organism,label,all families,strongest family,its per-class recall,share of all,flagged
0,Tg,compartment,0.33,gse19092,0.19,0.59,none
1,Tg,lopit_unified,0.49,gse108740,0.32,0.64,none
2,Pf,lopit_pf_location,0.47,plasmodb_pf3d7_expression,0.32,0.68,none


## 3. Certainty that means what it says

Support mapped to the held-out rate of being right (isotonic), with its calibration error measured out of fold. A recipe whose certainty is not calibrated makes no claims.

In [4]:
pd.DataFrame([{'strategy': k, 'n': (c := C.certainty_model(tg, k)).n, 'calibration error': c.calibration_error,
               'calibrated': c.calibrated, 'certainty at support 0.3/0.6/0.9': tuple(np.round(c([.3, .6, .9]), 2))}
              for k in ('feature_knn', 'supervised_classifier', 'random_forest', 'physical_partners')]).round(3)

,strategy,n,calibration error,calibrated,certainty at support 0.3/0.6/0.9
0,feature_knn,2845,0.024,True,"(0.25, 0.61, 0.89)"
1,supervised_classifier,3804,0.025,True,"(0.3, 0.57, 0.82)"
2,random_forest,3804,0.014,True,"(0.49, 0.97, 0.97)"
3,physical_partners,1159,0.032,True,"(0.31, 0.57, 0.6)"


Calibration held across tachyzoite expression quartiles and across distance from the labelled genes (measured 2026-10-04, ECE 0.03-0.05) -- but 42% of unlabelled genes on compartment lie farther from any labelled gene than 95% of labelled genes do. Claims for those carry no certainty ('outside tested range').

## 4. Recipes: the generator independent evidence can test

For each generator: the verifiers measured independent of it, the calibration error after verification, and on held-out genes how many claims reached 80% and how often they were right.

In [5]:
C.candidates('Tg', 'compartment').round(3)

,generator,verifiers,testable,certainty_error,answered,verified_error,confident,confident_right,confident_precision,calibrated
0,random_forest,"(physical_partners,)",True,0.014,3804,0.018,442,387,0.876,True
1,physical_partners,"(random_forest,)",True,0.032,1159,0.019,418,371,0.888,True
2,supervised_classifier,"(physical_partners,)",True,0.025,3804,0.031,389,334,0.859,True
3,feature_knn,"(physical_partners, structural_homology)",True,0.024,2845,0.022,309,271,0.877,True
4,map_neighbours,"(physical_partners, structural_homology)",True,0.046,1374,0.048,98,86,0.878,True
5,layer_vote,"(structural_homology,)",True,0.019,3804,0.027,0,0,NaN,True
6,structural_homology,(),False,0.045,1046,NaN,NaN,NaN,NaN,NaN


## 5. The shipped claims

Per label: the recipe, whether it is proven (calibrated end to end), and what it claims about genes with no label. Tested = an independent check agreed or disagreed.

In [6]:
rec = C.recipes()
rec[['organism', 'target', 'generator', 'verifiers', 'proven', 'heldout_confident_precision', 'claims']].round(2)

,organism,target,generator,verifiers,proven,heldout_confident_precision,claims
0,Tg,compartment,random_forest,physical_partners,True,0.88,4313
1,Tg,lopit_unified,random_forest,physical_partners,True,0.88,4313
2,Tg,dtm_class,random_forest,"physical_partners, structural_homology, layer_propagation, feature_knn, map_neighbours",True,0.96,0
3,Tg,screen_any_phenotype,feature_knn,"physical_partners, structural_homology, layer_propagation, supervised_classifier, random_forest, map_neighbours",False,0.47,0
4,Tg,cellcycle_phase,feature_knn,,True,NaN,7110
5,Tg,screen_actin_phenotype,feature_knn,"physical_partners, structural_homology, layer_propagation, supervised_classifier, random_forest, map_neighbours",False,0.87,0
6,Tg,screen_apicoplast_phenotype,feature_knn,"physical_partners, layer_propagation, supervised_classifier, random_forest, map_neighbours",False,0.8,0
7,Tg,screen_egress_phenotype,feature_knn,"layer_propagation, supervised_classifier, random_forest, map_neighbours",False,0.89,0
8,Tg,screen_replication_phenotype,feature_knn,"physical_partners, structural_homology, layer_propagation, supervised_classifier, random_forest, map_neighbours",False,0.86,0
9,Pf,lopit_pf_location,map_neighbours,structural_homology,True,0.88,620


In [7]:
cl = C.shipped()
cl.groupby(['organism', 'target', 'status'], observed=True).size().unstack(fill_value=0)

status                             outside tested range  tested  untested
organism target                                                          
Pf       lopit_pf_location                           28      68       524
         pb_transferred_phenotype                   524    2696        65
Tg       cellcycle_phase                            263       0      6847
         compartment                               1804     189      2320
         lopit_unified                             1804     189      2320

Discoveries: tested, confidence at least 0.8 and at least twice the claimed class's prior.

In [8]:
d = pd.concat([C.discoveries(o) for o in ('Tg', 'Pf')])
d.groupby(['organism', 'target'], observed=True).size()

organism  target                  
Pf        lopit_pf_location            13
          pb_transferred_phenotype    667
Tg        compartment                  13
          lopit_unified                30

In [9]:
ctx = S.Context.shipped('Tg')
top = C.discoveries('Tg').head(15)
top.assign(product=ctx.product(np.asarray([ctx.index[g.upper()] for g in top.gene_id.astype(str)])))[
    ['gene_id', 'product', 'target', 'claim', 'certainty', 'confidence', 'prior', 'lift']].round(2)

,gene_id,product,target,claim,certainty,confidence,prior,lift
0,TGME49_295658,zinc finger in N-recognin protein,lopit_unified,nucleus,0.94,0.99,0.33,2.98
1,TGME49_228670,"zinc finger, C2H2 type domain-containing protein",lopit_unified,nucleus,0.89,0.97,0.33,2.94
2,TGME49_295658,zinc finger in N-recognin protein,compartment,nucleus - chromatin,0.85,0.96,0.2,4.75
3,TGME49_203358,hypothetical protein,lopit_unified,nucleus,0.82,0.95,0.33,2.88
4,TGME49_289920,hypothetical protein,lopit_unified,apical_secretory,0.77,0.94,0.12,7.99
5,TGME49_263680,hypothetical protein,lopit_unified,mitochondrion,0.76,0.93,0.12,7.63
6,TGME49_242415,histone lysine-specific demethylase,lopit_unified,nucleus,0.76,0.93,0.33,2.83
7,TGME49_286270,hypothetical protein,compartment,nucleus - chromatin,0.97,0.93,0.2,4.6
8,TGME49_294060,hypothetical protein,lopit_unified,nucleus,0.73,0.92,0.33,2.79
9,TGME49_224880,kinesin motor domain-containing protein,lopit_unified,apical_secretory,0.73,0.92,0.12,7.87
